In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from db import engine

plt.rcParams["figure.figsize"] = (15, 6)
sns.set_style('whitegrid')

In [2]:
with engine.connect() as conn:
    loc_df = pd.read_sql("select * from location", conn)
    weather_df = pd.read_sql("select * from weather_observation", conn)

In [3]:
def to_f(temp):
    return (temp * (9/5)) + 32

In [4]:
df = loc_df.merge(weather_df, left_on='id', right_on='location_id', suffixes=['_loc', '_wea'])

In [5]:
df['observed_date'] = df['observed_at'].dt.strftime('%Y-%m-%d')

In [6]:
df['year'] = df['observed_at'].dt.year

In [7]:
df['month'] = df['observed_at'].dt.month_name()

In [8]:
df['month'] = pd.Categorical(df['month'], categories=['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December'], ordered=True)

In [9]:
df['temperature_f'] = df['temperature'].apply(to_f)
df['feels_like_f'] = df['feels_like'].apply(to_f)

## Dataset Overview

In [10]:
df['city'].nunique()

30

In [11]:
df['observed_at'].min()

Timestamp('2023-01-01 00:00:00+0000', tz='UTC')

In [12]:
df['observed_at'].max()

Timestamp('2025-12-31 23:00:00+0000', tz='UTC')

In [13]:
df.shape

(789120, 24)

In [14]:
df['city'].value_counts()

city
Seattle           26304
Bothell           26304
Portland          26304
Spokane Valley    26304
Olympia           26304
Bellingham        26304
Yakima            26304
Bend              26304
San Francisco     26304
Los Angeles       26304
San Diego         26304
Phoenix           26304
Las Vegas         26304
Albuquerque       26304
Denver            26304
Salt Lake City    26304
Boise             26304
Chicago           26304
Minneapolis       26304
Kansas City       26304
Dallas            26304
Houston           26304
New Orleans       26304
Atlanta           26304
Miami             26304
New York          26304
Boston            26304
Philadelphia      26304
Anchorage         26304
Honolulu          26304
Name: count, dtype: int64

In [15]:
df.isna().sum()

id_loc                      0
city                        0
state                       0
country                     0
latitude                    0
longitude                   0
created_at_loc              0
id_wea                      0
location_id                 0
observed_at                 0
temperature                 0
feels_like                  0
humidity                    0
pressure                    0
wind_speed                  0
weather_description     12087
weather_code                0
raw_json               789120
created_at_wea              0
observed_date               0
year                        0
month                       0
temperature_f               0
feels_like_f                0
dtype: int64

In [16]:
df['weather_description'].value_counts(dropna=False)

weather_description
Partly cloudy       366469
Clear               358148
Drizzle              22281
Foggy                16296
NaN                  12087
Rain                  7387
Snow                  5611
Rain Showers           698
Freezing Drizzle       106
Freezing Rain           34
Snow Showers             3
Name: count, dtype: int64

> From the above outputs, we see that we have 30 U.S. cities, with hourly weather data ranging from January 1st, 2023 to December 31st, 2025. We have 3 years worth of data, with each city having the same amount of hourly data. 

> The only columns in which there is missing data is the raw_json, which we will drop as this is missing for every row, and weather_description. The missing data in weather description means that the data contained a weather code that was not covered in the data dictionary during the transformation process. 

In [17]:
df[df['weather_description'].isna()].value_counts('weather_code')

weather_code
95    12087
Name: count, dtype: int64

Looking at the OpenMeteo documentation, the weather code "95" corresponds to thunderstorms, so we can fill in the missing values for weather descriptions

In [18]:
df.loc[
    (df['weather_description'].isna()) & (df['weather_code'] == 95), "weather_description"
] = "Thunderstorm"

In [22]:
df.drop(columns=['raw_json'], inplace=True)

In [23]:
df.isna().sum()

id_loc                 0
city                   0
state                  0
country                0
latitude               0
longitude              0
created_at_loc         0
id_wea                 0
location_id            0
observed_at            0
temperature            0
feels_like             0
humidity               0
pressure               0
wind_speed             0
weather_description    0
weather_code           0
created_at_wea         0
observed_date          0
year                   0
month                  0
temperature_f          0
feels_like_f           0
dtype: int64

In [24]:
df['weather_description'].value_counts(dropna=False)

weather_description
Partly cloudy       366469
Clear               358148
Drizzle              22281
Foggy                16296
Thunderstorm         12087
Rain                  7387
Snow                  5611
Rain Showers           698
Freezing Drizzle       106
Freezing Rain           34
Snow Showers             3
Name: count, dtype: int64

> As we can see, we now have no missing data in the weather_description column and we have dropped the raw_json column entirely. 

In [25]:
region_map = {
    "Seattle": "Pacific Northwest",
    "Bothell": "Pacific Northwest",
    "Portland": "Pacific Northwest",
    "Spokane Valley": "Pacific Northwest",
    "Olympia": "Pacific Northwest",
    "Bellingham": "Pacific Northwest",
    "Yakima": "Pacific Northwest",
    "Bend": "Pacific Northwest",

    "San Francisco": "West Coast",
    "Los Angeles": "West Coast",
    "San Diego": "West Coast",

    "Phoenix": "Southwest",
    "Las Vegas": "Southwest",
    "Albuquerque": "Southwest",

    "Denver": "Mountain West",
    "Salt Lake City": "Mountain West",
    "Boise": "Mountain West",

    "Chicago": "Midwest",
    "Minneapolis": "Midwest",
    "Kansas City": "Midwest",

    "Dallas": "South",
    "Houston": "South",
    "New Orleans": "South",
    "Atlanta": "South",
    "Miami": "South",

    "New York": "Northeast",
    "Boston": "Northeast",
    "Philadelphia": "Northeast",

    "Anchorage": "Alaska/Hawaii",
    "Honolulu": "Alaska/Hawaii",
}

In [28]:
df['region'] = df.apply(lambda x: region_map[x['city']], axis=1)

In [29]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 789120 entries, 0 to 789119
Data columns (total 24 columns):
 #   Column               Non-Null Count   Dtype              
---  ------               --------------   -----              
 0   id_loc               789120 non-null  object             
 1   city                 789120 non-null  str                
 2   state                789120 non-null  str                
 3   country              789120 non-null  str                
 4   latitude             789120 non-null  float64            
 5   longitude            789120 non-null  float64            
 6   created_at_loc       789120 non-null  datetime64[us, UTC]
 7   id_wea               789120 non-null  object             
 8   location_id          789120 non-null  object             
 9   observed_at          789120 non-null  datetime64[us, UTC]
 10  temperature          789120 non-null  float64            
 11  feels_like           789120 non-null  float64            
 12  humidity     

In [36]:
df['weather_code'] = df['weather_code'].astype('category')
df['weather_description'] = df['weather_description'].astype('category')
df['region'] = df['region'].astype('category')
month_order = [
    'January', 'February', 'March', 'April', 'May', 'June',
    'July', 'August', 'September', 'October', 'November', 'December'
]

df['month'] = pd.Categorical(
    df['month'],
    categories=month_order,
    ordered=True
)

In [37]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 789120 entries, 0 to 789119
Data columns (total 24 columns):
 #   Column               Non-Null Count   Dtype              
---  ------               --------------   -----              
 0   id_loc               789120 non-null  object             
 1   city                 789120 non-null  str                
 2   state                789120 non-null  str                
 3   country              789120 non-null  str                
 4   latitude             789120 non-null  float64            
 5   longitude            789120 non-null  float64            
 6   created_at_loc       789120 non-null  datetime64[us, UTC]
 7   id_wea               789120 non-null  object             
 8   location_id          789120 non-null  object             
 9   observed_at          789120 non-null  datetime64[us, UTC]
 10  temperature          789120 non-null  float64            
 11  feels_like           789120 non-null  float64            
 12  humidity     

In [38]:
def get_season(month):
    if month in ['December', 'January', 'February']:
        return 'Winter'
    elif month in ['March', 'April', 'May']:
        return 'Spring'
    elif month in ['June', 'July', 'August']:
        return 'Summer'
    elif month in ['September', 'October', 'November']:
        return 'Fall'
    else:
        return None

In [40]:
df['season'] = df.apply(lambda row: get_season(row['month']), axis=1)

In [43]:
season_order = ['Fall', 'Winter', 'Spring', 'Summer']

df['season'] = pd.Categorical(df['season'], categories=season_order, ordered=True)

In [34]:
df.groupby('region')['temperature_f'].mean().sort_values(ascending=False)

region
South                70.124424
Southwest            69.055534
West Coast           60.995472
Alaska/Hawaii        56.310795
Northeast            55.305878
Mountain West        54.657655
Midwest              52.654496
Pacific Northwest    52.234496
Name: temperature_f, dtype: float64

In [44]:
df.groupby('season')['temperature_f'].mean().sort_values(ascending=False)

season
Summer    74.425486
Fall      60.014549
Spring    56.776421
Winter    43.066669
Name: temperature_f, dtype: float64

In [45]:
pd.pivot_table(data=df, columns='season', index='region', values='temperature_f')

season,Fall,Winter,Spring,Summer
region,,,,
Alaska/Hawaii,56.827294,45.774815,54.931821,67.523995
Midwest,55.980302,29.388699,51.490380,73.373270
Mountain West,55.561273,34.837426,51.662237,76.220444
Northeast,57.292958,35.708524,53.380199,74.508406
Pacific Northwest,52.449004,38.537236,50.085720,67.620217
South,71.523555,56.111332,69.881766,82.742391
Southwest,70.076941,49.176172,66.607944,90.012047
West Coast,64.279212,54.754474,57.970707,66.900127


## Monthly Trends

### Top 5 Hottest Cities on Average

In [55]:
df.groupby('city')['temperature_f'].mean().sort_values(ascending=False)[:5]

city
Phoenix        76.421241
Miami          76.256337
Honolulu       75.261341
New Orleans    70.920812
Houston        70.815682
Name: temperature_f, dtype: float64

### Top 5 Coldest Cities on Average

In [54]:
df.groupby('city')['temperature_f'].mean().sort_values(ascending=True)[:5]

city
Anchorage         37.360249
Minneapolis       49.218577
Bend              49.764147
Spokane Valley    50.117475
Chicago           51.128894
Name: temperature_f, dtype: float64

## Season Swings

In [58]:
seasonal_swings = pd.pivot_table(data=df, columns='season', index='city', values='temperature_f').reset_index()

In [59]:
seasonal_swings

season,city,Fall,Winter,Spring,Summer
0,Albuquerque,60.694588,40.221439,58.345462,80.607500
1,Anchorage,36.419698,19.173865,36.148668,57.359076
2,Atlanta,63.965934,47.244068,63.397880,77.961984
3,Bellingham,52.249670,40.981208,50.253424,63.790380
4,Bend,50.209725,34.217703,45.904212,68.448152
5,Boise,55.762170,35.564852,52.355353,76.789130
6,Boston,54.290714,32.679760,49.906658,72.055489
7,Bothell,53.396731,41.525138,51.093750,66.195000
8,Chicago,54.706429,30.022380,48.585435,70.857853
9,Dallas,71.611786,50.519963,68.522826,86.693152


In [66]:
seasonal_swings['fall > winter'] = (seasonal_swings['Winter'] - seasonal_swings['Fall'])
seasonal_swings['winter > spring'] = (seasonal_swings['Spring'] - seasonal_swings['Winter'])
seasonal_swings['spring > summer'] = (seasonal_swings['Summer'] - seasonal_swings['Spring'])
seasonal_swings['summer > fall'] = (seasonal_swings['Fall'] - seasonal_swings['Summer'])

In [67]:
seasonal_swings.head()

season,city,Fall,Winter,Spring,Summer,fall > winter,winter > spring,spring > summer,summer > fall
0,Albuquerque,60.694588,40.221439,58.345462,80.607500,-20.473149,18.124023,22.262038,-19.912912
1,Anchorage,36.419698,19.173865,36.148668,57.359076,-17.245832,16.974803,21.210408,-20.939378
2,Atlanta,63.965934,47.244068,63.397880,77.961984,-16.721866,16.153812,14.564103,-13.996050
3,Bellingham,52.249670,40.981208,50.253424,63.790380,-11.268462,9.272215,13.536957,-11.540710
4,Bend,50.209725,34.217703,45.904212,68.448152,-15.992022,11.686509,22.543940,-18.238427


In [68]:
seasonal_swings = seasonal_swings.round(2)

In [70]:
seasonal_swings

season,city,Fall,Winter,Spring,Summer,fall > winter,winter > spring,spring > summer,summer > fall
0,Albuquerque,60.69,40.22,58.35,80.61,-20.47,18.12,22.26,-19.91
1,Anchorage,36.42,19.17,36.15,57.36,-17.25,16.97,21.21,-20.94
2,Atlanta,63.97,47.24,63.40,77.96,-16.72,16.15,14.56,-14.00
3,Bellingham,52.25,40.98,50.25,63.79,-11.27,9.27,13.54,-11.54
4,Bend,50.21,34.22,45.90,68.45,-15.99,11.69,22.54,-18.24
5,Boise,55.76,35.56,52.36,76.79,-20.20,16.79,24.43,-21.03
6,Boston,54.29,32.68,49.91,72.06,-21.61,17.23,22.15,-17.76
7,Bothell,53.40,41.53,51.09,66.19,-11.87,9.57,15.10,-12.80
8,Chicago,54.71,30.02,48.59,70.86,-24.68,18.56,22.27,-16.15
9,Dallas,71.61,50.52,68.52,86.69,-21.09,18.00,18.17,-15.08
